In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

In [6]:
#setting up the spark session

spark = SparkSession.builder \
    .appName("Yellow Taxi Analysis") \
    .master("local[*]") \
    .getOrCreate()

# Display the Spark version
print("Spark version:", spark.version)

Spark version: 3.4.1


In [7]:
# setting up the df 
df = spark.read.parquet("/home/dineshswain2001/data/yellow_tripdata_2024-10.parquet")

#question 1
df.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [8]:
#question2

df_repartitioned = df.repartition(4)

output_path = "/home/dineshswain2001/data/yellow_tripdata_repartitioned"
df_repartitioned.write.mode('overwrite').parquet(output_path)

import os

parquet_files = [f for f in os.listdir(output_path) if f.endswith('.parquet')]

total_size_bytes = sum(os.path.getsize(os.path.join(output_path, f)) for f in parquet_files)
avg_size_mb = (total_size_bytes / len(parquet_files)) / (1024 * 1024)

print(f"Number of parquet files: {len(parquet_files)}")
print(f"Average size of parquet files: {avg_size_mb:.2f} MB")

[Stage 5:=============================>                             (2 + 2) / 4]

Number of parquet files: 4
Average size of parquet files: 23.04 MB


In [9]:
#question 3
from pyspark.sql.functions import dayofmonth, month, year

oct_15_trips = df.filter(
    (year(df.tpep_pickup_datetime) == 2024) & 
    (month(df.tpep_pickup_datetime) == 10) & 
    (dayofmonth(df.tpep_pickup_datetime) == 15)
)

trip_count = oct_15_trips.count()

print(f"Number of taxi trips on October 15th, 2024: {trip_count}")

[Stage 6:=============================>                             (1 + 1) / 2]

Number of taxi trips on October 15th, 2024: 128893


In [10]:
#quetion 4
from pyspark.sql.functions import col, unix_timestamp, max

df_with_duration = df.withColumn(
    "trip_duration_seconds", 
    unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))
)

# Convert to hours
df_with_duration = df_with_duration.withColumn(
    "trip_duration_hours", 
    col("trip_duration_seconds") / 3600
)

max_duration = df_with_duration.select(max("trip_duration_hours")).collect()[0][0]

print(f"Length of the longest trip in hours: {max_duration:.2f}")

[Stage 9:>                                                          (0 + 2) / 2]

Length of the longest trip in hours: 162.62


In [11]:
#question 6
zones_df = spark.read.option("header", "true").csv("/home/dineshswain2001/data/taxi_zone_lookup.csv")

zones_df.createOrReplaceTempView("zones")

df.createOrReplaceTempView("trips")

print("Zones schema:")
zones_df.printSchema()

print("Trips schema:")
df.printSchema()


query = """
SELECT z.Zone, COUNT(*) as trip_count
FROM trips t
JOIN zones z ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trip_count ASC
LIMIT 1
"""

least_frequent_zone = spark.sql(query)
least_frequent_zone.show(truncate=False)

Zones schema:
root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Trips schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge:

[Stage 14:==========================================================(2 + 0) / 2]

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
+---------------------------------------------+----------+

